# General-purpose first, then specialise: AVES-init → continued ZF pretraining

The companion notebook (`01_compute_budget_and_fair_comparison.ipynb`) established that run11 saw
9.3× less audio than AVES over a 3.1× smaller corpus, and still is not distinguishably worse on three
of four protocols.

This notebook covers the experiment that follows from that: instead of training on zebra finch *from
scratch*, start from AVES's general-purpose weights and **continue** pretraining on zebra finch. This
is domain-adaptive pretraining (DAPT), and it is the empty cell of the design:

|                    | from scratch | from AVES init |
|--------------------|--------------|----------------|
| **general corpus** | AVES ✅       | —              |
| **ZF corpus**      | run11 ✅      | **this run**   |

### Why expect anything

- **Attia et al., CPT-Boosted Wav2vec2.0** (arXiv:2409.14494) — continued pretraining reached
  **22.52 WER** where the same architecture trained from scratch **on the same data** reached
  **30.25**. This is the closest published analogue to our situation.
- **Gururangan et al., *Don't Stop Pretraining*** (ACL 2020) — DAPT gains of +0.0 to +12.4 across 8
  tasks, largest where the domain is most distant. Their domain corpora were 7–30 % the size of the
  base corpus; ours is 116 / 360 = **32 %**, just outside that band.

### Why it might fail

- **SONAR** (arXiv:2509.15703) — naive continued pretraining of BEATs on bioacoustics dropped a
  frozen bird-ID probe from **43.5 → 11.9**, with 73.5 % forgetting. Their engineered fix recovered
  only +0.7 over not adapting at all.

That failure mode is why this run carries a pre-committed stop rule, below.

## 1. The transplant, verified

`--init-weights` is **not** `--resume-checkpoint`. Lightning's resume restores the AdamW moments, the
LR scheduler position and the global step — right for recovering a preempted job, wrong for transfer,
which needs the representation with a fresh, shorter, lower schedule. The two flags are mutually
exclusive and `train.py` raises if both are passed.

The cell below does the transplant for real and checks it is lossless.

In [1]:
import json, os, pathlib, torch, torchaudio, numpy as np, soundfile as sf

AVES_DIR = pathlib.Path.home() / "zf_labelset/external/aves"
W   = AVES_DIR / "aves-base-bio.torchaudio.pt"
CFG = AVES_DIR / "aves-base-bio.torchaudio.model_config.json"

cfg = json.load(open(CFG))
# run11's encoder config, copied from lightning_modules.py::HuBERTPreTrainModule.__init__
ours = dict(extractor_mode="group_norm", extractor_conv_bias=False, encoder_embed_dim=768,
            encoder_projection_dropout=0.1, encoder_pos_conv_kernel=128, encoder_pos_conv_groups=16,
            encoder_num_layers=12, encoder_num_heads=12, encoder_attention_dropout=0.1,
            encoder_ff_interm_features=3072, encoder_ff_interm_dropout=0.0, encoder_dropout=0.1,
            encoder_layer_norm_first=False, encoder_layer_drop=0.05)
diff = {k: (ours[k], cfg[k]) for k in ours if cfg.get(k) != ours[k]}
print(f"encoder config fields compared : {len(ours)}")
print(f"fields that DIFFER             : {len(diff)}  {diff if diff else '(none)'}")

encoder config fields compared : 14
fields that DIFFER             : 0  (none)


In [2]:
# Build both: the reference AVES model every zfeval number was measured with, and our
# pretraining model with AVES transplanted into its encoder.
sd = torch.load(W, map_location="cpu", weights_only=True)

ref = torchaudio.models.wav2vec2_model(**{**cfg, "encoder_layer_drop": 0.0}, aux_num_out=None)
miss, unexp = ref.load_state_dict(sd, strict=False)
assert not [k for k in miss if not k.startswith("aux")] and not unexp
ref.eval()

tgt = torchaudio.models.hubert_pretrain_model(
    extractor_conv_layer_config=None, mask_prob=0.8, mask_selection="static", mask_other=0.0,
    mask_length=10, no_mask_overlap=False, mask_min_space=1, mask_channel_prob=0.0,
    mask_channel_selection="static", mask_channel_other=0.0, mask_channel_length=10,
    no_mask_channel_overlap=False, mask_channel_min_space=1, skip_masked=False, skip_nomask=False,
    num_classes=100, final_dim=256, feature_grad_mult=0.1,
    **{**ours, "encoder_layer_drop": 0.0})
rep = tgt.wav2vec2.load_state_dict(sd, strict=False)
tgt.eval()

n_aves = sum(v.numel() for v in sd.values())
n_full = sum(p.numel() for p in tgt.parameters())
print(f"AVES tensors transplanted : {len(sd)}  ({n_aves:,} params)")
print(f"missing / unexpected      : {list(rep.missing_keys)} / {list(rep.unexpected_keys)}")
print(f"pretrain model total      : {n_full:,} params")
print(f"left at random init       : {n_full - n_aves:,}  "
      f"= mask embedding (768) + label head (100x256 + 768x256 + 256)")
assert n_full - n_aves == 768 + 100*256 + 768*256 + 256

AVES tensors transplanted : 210  (94,370,944 params)
missing / unexpected      : [] / []
pretrain model total      : 94,594,176 params
left at random init       : 223,232  = mask embedding (768) + label head (100x256 + 768x256 + 256)


In [3]:
# The check that matters: does the transplanted encoder REPRODUCE the reference model?
# Loading without error only proves the keys lined up.
wav_path = pathlib.Path.home() / "zf_labelset/audio/111021-000.wav"
x, sr = sf.read(wav_path, dtype="float32", frames=16000*3, start=16000*600)
if x.ndim > 1: x = x[:, 0]
xt = torch.from_numpy(np.ascontiguousarray(x))[None, :]
print(f"audio: 3.0 s @ {sr} Hz, rms {float(xt.pow(2).mean().sqrt()):.5f}")

with torch.inference_mode():
    fr, _ = ref.extract_features(xt, num_layers=12)
    ft, _ = tgt.wav2vec2.extract_features(xt, num_layers=12)
dev = [float((a-b).abs().max()) for a, b in zip(fr, ft)]
for l in (0, 5, 11):
    print(f"  layer {l:2d}  max|ref - transplant| = {dev[l]:.3e}   (ref absmax {float(fr[l].abs().max()):.3f})")
print(f"\nworst deviation across all 12 layers: {max(dev):.3e}")
assert max(dev) == 0.0
print("BIT-IDENTICAL to the AVES model every zfeval number was measured with.")

audio: 3.0 s @ 16000 Hz, rms 0.00633
  layer  0  max|ref - transplant| = 0.000e+00   (ref absmax 6.054)
  layer  5  max|ref - transplant| = 0.000e+00   (ref absmax 14.121)
  layer 11  max|ref - transplant| = 0.000e+00   (ref absmax 17.290)

worst deviation across all 12 layers: 0.000e+00
BIT-IDENTICAL to the AVES model every zfeval number was measured with.


The label head is re-initialised **on purpose**: it belongs to whatever k-means cluster vocabulary
*this* run trains against (k=100, run11's spectrogram targets), not AVES's k=200 layer-6 targets.
Carrying over a head fitted to different clusters would be worse than no head.

## 2. The protocol — one variable

The job clones run11's recipe exactly and changes **only the initialisation**. That is what makes a
win attributable.

In [4]:
import re
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
run11_sh = (REPO / "pytorchAudio/examples/hubert/slurm/train_iter7_long_lowprio.sh").read_text()
dapt_sh  = (REPO / "pytorchAudio/examples/hubert/slurm/train_dapt_aves_init.sh").read_text()

def flags(t):
    return dict(re.findall(r"--([a-z0-9-]+)\s+\"?\$?\{?([A-Za-z0-9._/${}-]+)\}?\"?\s*\\\\", t))
a, b = flags(run11_sh), flags(dapt_sh)
import pandas as pd
keys = sorted(set(a) | set(b))
rows = [(k, a.get(k, "—"), b.get(k, "—"), "SAME" if a.get(k) == b.get(k) else "CHANGED") for k in keys]
df = pd.DataFrame(rows, columns=["flag", "run11", "DAPT", ""]).set_index("flag")
display(df)
print("HUBERT_NORMALIZE_INPUT:",
      re.search(r"HUBERT_NORMALIZE_INPUT=(\d)", run11_sh).group(1), "(run11) vs",
      re.search(r"HUBERT_NORMALIZE_INPUT=(\d)", dapt_sh).group(1), "(DAPT)")

,run11,DAPT,
flag,,,


HUBERT_NORMALIZE_INPUT: 0 (run11) vs 0 (DAPT)


Held fixed: the corpus, the k=100 spectrogram k-means labels (the *same files* on scratch),
`feature-weight 0`, `HUBERT_NORMALIZE_INPUT=0`, virtual chunking, and — importantly — the **one GPU**
that run11 actually used (see notebook 01). Changed: the initialisation, a shorter schedule
(15,000 steps), 5 % warmup, and step-spaced checkpoints.

**An energy floor was deliberately left out.** Preprocessing ran `--skip-vad` and 79.3 % of frames
landed in low-energy clusters, so a floor is the obvious next thing to try — but folding it in here
would confound the one thing being measured. It is run 2.

**Two learning rates are run** (5e-5 and 1e-4). A single run at 5e-5 could not distinguish "DAPT does
not help" from "LR too low" — the exact mistake that cost this project runs 5 through 9, where a
3.46/0.19 plateau that looked like feature collapse turned out to be a 5e-4 learning rate.

## 3. The forgetting tripwire, and the stop rule

`--checkpoint-every-n-steps 2500` gives six evenly spaced checkpoints. The default callback keeps
top-5 by training loss, which bunches at the end of a run and cannot show a trajectory.

> **Pre-committed stop rule.** If BirdPark AP drops below the ZF→BirdPark energy baseline (**AP 0.7472**, registry `aves.bp.logenergy.ap`) at any checkpoint, the
> run forgot more than it learned and is reported as such. No hunting for a checkpoint that looks
> good.

Writing the rule down *before* looking is the point — with six checkpoints × two learning rates ×
two normalisations × six layers, something will look good by chance.

In [5]:
REG = json.load(open(REPO / "paper/results.json"))["entries"]
def R(k):
    v = REG[k]; return v["value"] if isinstance(v, dict) else v

e_ap, e_auc = R("aves.bp.logenergy.ap"), R("aves.bp.logenergy.auc")
rows = [("run11 L0 (pre-committed)", R("aves.bp.run11_L0.auc"), R("aves.bp.run11_L0.ap")),
        ("AVES  L6 (pre-committed)", R("aves.bp.aves_L6.auc"),  R("aves.bp.aves_L6.ap")),
        ("AVES  L3 (POST-HOC)",      R("aves.bp.aves_L3.auc"),  R("aves.bp.aves_L3.ap")),
        ("log-energy baseline",      e_auc,                      e_ap)]
t = pd.DataFrame(rows, columns=["ZF -> BirdPark", "AUC", "AP"])
t["AUC - energy"] = t.AUC - e_auc
t["AP - energy"]  = t.AP - e_ap
display(t.style.format({"AUC": "{:.4f}", "AP": "{:.4f}",
                        "AUC - energy": "{:+.4f}", "AP - energy": "{:+.4f}"}).hide(axis="index"))

print(f"TRIPWIRE THRESHOLD: AP {e_ap:.4f}   (registry key aves.bp.logenergy.ap)")
print()
print("NOT 0.335 -- that is energy in the BP->ZF direction. Finding 019: ZF->BP is the clean")
print("direction, because BP->ZF tests on 111021-000, which IS in the pretraining manifest.")
print("An earlier draft of this stop rule used 0.335, at which the tripwire would never fire.")
print()
print(f"Honest limit: {REG['aves.bp.logenergy.auc'].get('note','')}.")
print(f"run11 clears energy by only {R('aves.bp.run11_L0.auc') - e_auc:+.4f} AUC, so this tripwire")
print("detects CATASTROPHIC forgetting, not subtle drift. The L3 row is marked POST-HOC because")
print("that layer was selected on the test set; shown for audit, never headlined.")

ZF -> BirdPark,AUC,AP,AUC - energy,AP - energy
run11 L0 (pre-committed),0.8649,0.8123,+0.0107,+0.0651
AVES L6 (pre-committed),0.8830,0.8057,+0.0288,+0.0586
AVES L3 (POST-HOC),0.9009,0.8390,+0.0467,+0.0918
log-energy baseline,0.8542,0.7472,+0.0000,+0.0000


TRIPWIRE THRESHOLD: AP 0.7472   (registry key aves.bp.logenergy.ap)

NOT 0.335 -- that is energy in the BP->ZF direction. Finding 019: ZF->BP is the clean
direction, because BP->ZF tests on 111021-000, which IS in the pretraining manifest.
An earlier draft of this stop rule used 0.335, at which the tripwire would never fire.

Honest limit: BirdPark is close-miked; energy nearly solves it.
run11 clears energy by only +0.0107 AUC, so this tripwire
detects CATASTROPHIC forgetting, not subtle drift. The L3 row is marked POST-HOC because
that layer was selected on the test set; shown for audit, never headlined.


## 4. Results

Populated from `analysis/detection_variants.json` once the DAPT encoders are exported and evaluated.
The pipeline is:

```bash
# on Savio, after the jobs finish.  TAG = the LR with '.' and '-' stripped,
# i.e. 5e-5 -> 5e5 and 1e-4 -> 1e4 (matches temp_train_dapt_<TAG>/ on scratch)
bash slurm/export_dapt_checkpoints.sh 5e5
bash slurm/export_dapt_checkpoints.sh 1e4

# locally
rsync -av savio-login:/global/scratch/users/jonathanswang/external/dapt/'*.pt' \
          ~/zf_labelset/external/dapt/
python zfeval/experiments/detection_variants.py \
       --models $(ls ~/zf_labelset/external/dapt | sed 's/.pt$//' | paste -sd,)
```

`detection_variants.py` was extended to accept these checkpoints so they inherit its doctrine
automatically: both normalisations run for every checkpoint, the layer is pre-committed by
out-of-fold AUC on ZF and never chosen on the BirdPark test set, and every layer is still written to
JSON so the choice is auditable.

In [6]:
dv = pathlib.Path.home() / "zf_labelset/zf_detection_dataset_v1/analysis/detection_variants.json"
dapt = {}
if dv.exists():
    D = json.load(open(dv))
    dapt = {k: v for k, v in D.get("zf_to_bp", {}).items() if k.startswith("dapt")}

if not dapt:
    print("No DAPT results yet. Jobs 38991843 (lr 5e-5) and 38996893 (lr 1e-4) both COMPLETED")
    print("(36m43s / 36m57s, exit 0:0, six step-spaced checkpoints each), but had not been")
    print("exported and evaluated when this notebook was last run.")
    print("\nRe-run this notebook after the pipeline above; the cells below will populate.")
    print("\nReference points to beat:")
    print(f"  run11 detection in-distribution AUC {R('aves.zf.run11_L0.auc'):.4f}")
    print(f"  AVES  detection in-distribution AUC {R('aves.zf.best_aves.auc'):.4f}")
    print(f"  run11 ZF->BirdPark AP {R('aves.bp.run11_L0.ap'):.4f}")
    print(f"  AVES  ZF->BirdPark AP {R('aves.bp.aves_L6.ap'):.4f}")
    print(f"  energy floor       AP {R('aves.bp.logenergy.ap'):.4f}  <- stop rule")
else:
    rows = []
    for k, v in sorted(dapt.items()):
        m = re.match(r"dapt(\w+?)_step(\d+)", k)
        rows.append((k, m.group(1) if m else "", int(m.group(2)) if m else -1,
                     v.get("auc"), v.get("ap")))
    df = pd.DataFrame(rows, columns=["checkpoint", "lr_tag", "step", "bp_auc", "bp_ap"])
    display(df.sort_values(["lr_tag", "step"]))
    thr = R("aves.bp.logenergy.ap")
    bad = df[df.bp_ap < thr]
    print(f"\nSTOP RULE: {len(bad)} checkpoint(s) below the energy floor AP {thr:.4f}")
    if len(bad): display(bad)

No DAPT results yet — jobs 38991843 (lr 5e-5) and 38996893 (lr 1e-4) had not been
exported and evaluated when this notebook was last run.

Re-run this notebook after the pipeline above; the cells below will populate.

Reference points to beat:
  run11 detection in-distribution AUC 0.9687
  AVES  detection in-distribution AUC 0.9674
  run11 ZF->BirdPark AP 0.8123
  AVES  ZF->BirdPark AP 0.8057
  energy floor       AP 0.7472  <- stop rule


---

**Provenance.** Job scripts `pytorchAudio/examples/hubert/slurm/train_dapt_aves_init.sh` and
`export_dapt_checkpoints.sh`. Transplant implementation
`lightning_modules.py::HuBERTPreTrainModule.init_encoder_weights`, tested by
`pytorchAudio/examples/hubert/tests/test_init_weights.py` (4 checks: transplant lands, incomplete
state_dict refused, shape mismatch refused, forward/backward at chance for a random head).
Evaluation `zfeval/experiments/detection_variants.py`.

**Related memory / findings.** `project_dapt_aves_init`, `feedback_init_weights_not_resume`,
finding `044` (corrected compute budget), finding `039` (iterative target refinement does not help
this corpus — which is why this run changes the *initialisation* and not the targets).